# Integrated Intelligent Systems: Tutorial 2 

| | |
|---|---|
| **Name** | *SHRINIHA PALANIYANDI ASHOKKUMAR* |
| **Matriculation no.** | *6570093* |
| **Program** | M.Sc. Artificial Intelligence and Intelligent Systems (AIIS), University of Bremen |
| **Semester** | Winter semester 2026/27 |
| **University email** | shrinih1@uni-bremen.de |
| **GitHub** | [github.com/shriniha](https://github.com/shriniha) |
| **Assignment** | Assignment 1: Foundations (Eva coffee scenario) |
| **Deadline** | 03.11.2026, 4 PM |

---

# Integrated Intelligent Systems: Tutorial 2 - Body Motion Problem (Action Designator & URDF) (3 Pts)

This notebook introduces the body motion problem central to designing and implementing Cognition-Enabled Robot Agents.

## Part 1: Action Designator (E1–E6) (1.5 Pt)

**Title:** The Body Motion Problem: Serving Eva a Drink & Breakfast

![Alternative Text](bmp.png)

### Scenario Background
Eva says: "Serve me a drink."

The kitchen contains two milk cartons in the fridge door:
- **Carton A:** Opened whole milk.
- **Carton B:** Unopened oat drink.

Eva's usual mug already contains hot black coffee on the counter.

A wet surface on Carton A causes slip during a low test lift, requiring a grasp adaptation before pouring.

---

### Multi-Question Student Task Sheet (0.25 Pt x 6)

1. **E1 · The Gap & Whole-Course Judgment**
   - Eva says "Serve me a drink." A video shows a robot executing a smooth trajectory, placing the mug on the table, but a photograph reveals milk spilled across the counter and an open fridge left behind. Explain why this nominal end-state success does **not** constitute a solution to the Body Motion Problem, and specify two whole-course constraints that were violated.

   **ANSWER:**
   - The Body Motion Problem is not just “reach the goal state.” It asks how the body moves through time while respecting constraints, so a correct end state reached by a bad path is not a solution. The mug ended up on the table, but the course of the action did damage the end-state check never saw. Two whole-course constraints were violated:
   1. Containment: the liquid must stay where it is meant to go during transport or pouring. The spilled milk breaks this.
   2. Environment restoration / side-effect freedom: things the robot changed to do the task (like opening the fridge) must be restored, and the
   workspace left clean. The open fridge and the mess break this.

2. **E2 · Constrained Action Designator & Grounding**

    - Distinguish interpretation, inferred means, and grounded objects for this episode by filling out the three missing slots in the schematic action designator:
    
    - **Working Interpretation:** `(:type serving) (:recipient Eva) (:substance milk)`
    - **Inferred Function/Means:** `(:vessel-role pouring-source)`
    - **Grounded Physical Object:** `(:object carton A)`

3. **E3 · The Commitment Decision & Asymmetric Failure Costs**

    - During the retrieval step, the robot faces a choice between a **5 mm test lift near the shelf** or a **high carry directly to the counter**. State the predicted effect, the cost of failure, and the single piece of evidence that justifies choosing the low test lift before committing to the carry.

    **ANSWER:**
    1. Predicted effect: a 5 mm lift transfers the carton’s weight to the gripper. If the grasp is adequate, the carton rises with no relative motion
    2. Cost of failure: a slip during the test lift means a drop of a few millimetres, which is cheap and recoverable. The same slip during a high carry means a dropped carton and spilled milk, and the fridge stays open longer. The costs are asymmetric: the test is cheap and the carry failure is expensive.
    3. Evidence: the wet surface on Carton A. It lowers friction, so the grasp’s holding ability is uncertain and needs checking before committing.

4. **E4 · Causal Prediction across Material Variants**

    - Compare pouring milk into coffee versus pancake batter into a pan. Formulate the stopping rule for each liquid using `(:stop-when ...)`, and explain how material dynamics alter when the robot must command the wrist to tilt back.

    **ANSWER:**
    1. Milk into coffee: (:stop-when (mug-level >= target-fill-line)) (or a target volume or colour mix). Milk is a thin, low-viscosity fluid, so flow starts and stops almost immediately with the tilt. The wrist can tilt back late and the overshoot is small.
    2. Batter into a pan: (:stop-when (batter-spread-diameter >= target-diameter)) (or poured mass at target). Batter is viscous, so it keeps flowing after the tilt starts to reverse and forms a thick stream that clings and drips. The robot must begin tilting back earlier, and may need a small twist or pause to cut the stream. The lag is longer, so the stopping rule has to anticipate the flow rather than react to it.

5. **E5 · System Decomposition & The Reasoning Loop**

    - Trace the carton slip event through the seven-stage reasoning loop: **Infer → Ground → Predict → Decide → Regulate → Detect & Revise → Verify**. State precisely which step was triggered when downward motion was observed during the 5 mm lift, and how the action designator was developed in response.

    **ANSWER:**
    1. Infer: serving means adding milk to the coffee, so the carton is a pouring source.
    2. Ground: the object is identified as Carton A.
    3. Predict: a high-side pinch should hold the carton, but the wet surface makes this uncertain.
    4. Decide: do a 5 mm test lift before committing to the carry.
    5. Regulate: execute the lift while monitoring the carton’s motion relative to the gripper.
    6. Detect & Revise: this stage is triggered when downward motion is observed during the lift. The observation contradicts the prediction of a stable hold. The designator is revised so the grasp parameters change, for example to a lower or wider grasp with more force, or two-sided support
    7. Verify: repeat the test lift with the revised grasp, and only on a successful hold proceed to carry and pour.


6. **E6 · Conceptual Ablation & Evidence of Competence**

    - Perform an ablation experiment. Suppose we remove the **"Check before commitment"** capability and replace it with a fixed default high-side-pinch lift. Predict a specific kitchen environment variant where this ablated system fails, and explain why single-run execution success differs from evidence of general competence across task variants.

    **ANSWER:**
    - With “Check before commitment” removed and a fixed high-side-pinch lift in its place, the system fails in a variant like a wet, condensation-covered carton. This is the same situation as the scenario, but now the slip happens during the carry. It also fails with a fuller or heavier carton, or a glass bottle. The result is a dropped carton and spilled milk, with nothing to detect the problem or recover.
    
    - A single successful run is one sample from one set of conditions. The fixed lift may succeed simply because the carton was dry and light, which says nothing about handling variation. Competence shows up across task variants: different surfaces, weights and materials, with the system detecting deviations and revising. Ablation shows which capability actually produces that robustness.

## Part 2: Kino-Dynamic Modeling of Robot World System with URDF (1.5 Pt)

**Title:**  Designing and Implementing the kino-dynamic model of the robot world system with URDF

### 2.1  Theoretical Background

In [10]:
from IPython.display import display, HTML
from pathlib import Path

html_path = Path("urdf_background.html")

html_content = html_path.read_text(encoding="utf-8")

display(HTML(html_content))

## 2.2  Hands-on: Model your own robot world system according to specifications below (1.5 Pt)

In [11]:
from IPython.display import display, HTML
from pathlib import Path

html_content = Path("urdf_handon.html").read_text(encoding="utf-8")

display(HTML(html_content))

**SOLUTION**

The .xacro files are stored inside the folder Eva-Robot

# Submission guidelines

- Deadline: 10.11.2026, 4 PM
- Submission as Git repo commits pushed within deadlines (If not yet, create a repo with the name **iis-assignment-win26** for the whole semester and invite me as collaborator)
- Put the textual answers in notebook and all related media (urdf, xacro, etc) should then be put into a folder called **Assignment2**. 